In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import missingno as msno
import os

# 1. CẤU HÌNH ĐƯỜNG DẪN
base_dir = os.path.abspath(os.path.join(os.getcwd(), ".."))
raw_path = os.path.join(base_dir, 'data', 'raw', 'accepted_2007_to_2018Q4.csv')
output_path = os.path.join(base_dir, 'data', 'processed', 'clean_loan.csv')

# 2. LOAD DỮ LIỆU (Lấy 100k dòng để đảm bảo máy Ngọc không bị treo)
print("--- Đang nạp dữ liệu ---")
df = pd.read_csv(raw_path, nrows=100000, low_memory=False)

# ---------------------------------------------------------
# BƯỚC 1: LOẠI BỎ DATA LEAKAGE (GIẢI QUYẾT LỖI 99%)
# ---------------------------------------------------------
# Chỉ giữ lại các cột có TRƯỚC khi giải ngân tiền
keep_list = [
    'loan_amnt', 'term', 'int_rate', 'installment', 'grade', 'sub_grade',
    'emp_length', 'home_ownership', 'annual_inc', 'verification_status',
    'purpose', 'addr_state', 'dti', 'delinq_2yrs', 'fico_range_low', 
    'fico_range_high', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 
    'total_acc', 'loan_status'
]
df = df[keep_list]

# ---------------------------------------------------------
# BƯỚC 2: XỬ LÝ NHÃN (LOAN_STATUS)
# ---------------------------------------------------------
# Chỉ lấy các khoản vay đã kết thúc để train (Tránh nhóm 'Current')
targets = ['Fully Paid', 'Charged Off', 'Default']
df = df[df['loan_status'].isin(targets)]
df['loan_status'] = df['loan_status'].map({'Fully Paid': 0, 'Charged Off': 1, 'Default': 1})

# ---------------------------------------------------------
# BƯỚC 3: CLEANING & FEATURE ENGINEERING
# ---------------------------------------------------------
# A. Xử lý cột 'term' (36 months -> 36)
df['term'] = df['term'].str.extract('(\d+)').astype(float)

# B. Xử lý cột 'emp_length' (< 1 year -> 0, 10+ years -> 10)
df['emp_length'] = df['emp_length'].str.extract('(\d+)').fillna(0).astype(float)

# C. Xử lý giá trị thiếu (Missing Values)
# Xóa cột nếu trống > 50%
df = df.dropna(thresh=len(df)*0.5, axis=1)

# Điền Median cho cột số, Mode cho cột chữ
for col in df.columns:
    if df[col].dtype == 'object':
        df[col] = df[col].fillna(df[col].mode()[0])
    else:
        df[col] = df[col].fillna(df[col].median())

# ---------------------------------------------------------
# BƯỚC 4: VISUALIZATION (KIỂM TRA SỨC KHỎE DỮ LIỆU)
# ---------------------------------------------------------
print("--- Đang vẽ biểu đồ phân tích ---")
plt.figure(figsize=(18, 5))

# Biểu đồ 1: Kiểm tra nhãn (Class Imbalance)
plt.subplot(1, 3, 1)
sns.countplot(x='loan_status', data=df, palette='viridis')
plt.title("Phân phối Nhãn (0: Tốt, 1: Xấu)")

# Biểu đồ 2: Kiểm tra lãi suất (Biến quan trọng nhất)
plt.subplot(1, 3, 2)
sns.boxplot(x='loan_status', y='int_rate', data=df)
plt.title("Chênh lệch Lãi suất")

# Biểu đồ 3: Ma trận tương quan (Heatmap)
plt.subplot(1, 3, 3)
sns.heatmap(df.corr(numeric_only=True), cmap='RdYlGn', annot=False)
plt.title("Tương quan giữa các biến")

plt.tight_layout()
plt.show()

# ---------------------------------------------------------
# BƯỚC 5: XUẤT FILE SẠCH
# ---------------------------------------------------------
df.to_csv(output_path, index=False)
print(f"✅ HOÀN THÀNH! Đã lưu file sạch tại: {output_path}")
print(f"Kích thước dữ liệu cuối cùng: {df.shape}")